# Isentropic diagnostics from precomputed conditional sums

Reads the files written by `scripts/sam_iso_integrals.py` (one per run, or several per run if the
snapshots were split between jobs) and makes the diagnostics of Pauluis and Mrowiec (2013):

1. time series of the overturning, to choose the averaging period,
2. the isentropic streamfunction $\Psi(z, \theta)$ in $\theta_e$ and frozen $\theta_{ef}$, with the
   two-stream mass fluxes $M^\pm$,
3. $\theta_e$ versus $\theta_{ef}$,
4. cloudy and clear air,
5. a comparison of all experiments.

Nothing here reads the 3D output, so the notebook runs in seconds. Use the **Python (iso)** kernel.

## Configuration

In [ ]:
from pathlib import Path

SUMS_DIR = Path("/scratch/op13/SAM-Torch/iso_sums")
PATTERN = "iso_sums_*.nc"

# Averaging period (days) for the time means; None = from the start / to the end
T_START, T_END = None, None

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import isentropic as iso

COORDS = ["theta_e", "theta_ef"]
LABEL = {"theta_e": r"$\theta_e$", "theta_ef": r"$\theta_{ef}$ (frozen)"}

## Read the conditional sums
Files with the same `experiment` attribute are joined in time (a time present twice is used once).
Experiments with fewer snapshots are padded with NaN.

In [ ]:
files = sorted(SUMS_DIR.glob(PATTERN))
assert files, f"no {PATTERN} in {SUMS_DIR}"
by_exp = {}
for f in files:
    ds = xr.open_dataset(f)
    by_exp.setdefault(ds.attrs["experiment"], []).append(ds)
    print(f"{f.name}: {ds.attrs['experiment']}, {ds.sizes['time']} snapshots, density {ds.attrs['density']}")

runs = []
for exp, parts in by_exp.items():
    for p in parts[1:]:
        assert all(p.attrs[f"bins_{c}"] == parts[0].attrs[f"bins_{c}"] for c in COORDS), f"{exp}: different bins"
    hist = sum(p.HIST_JOINT for p in parts)
    ds = xr.concat([p.drop_vars("HIST_JOINT") for p in parts], dim="time").sortby("time")
    dup = ds.get_index("time").duplicated(keep="last")
    if dup.any():
        print(f"{exp}: {dup.sum()} snapshots in several files, used once (HIST_JOINT counts them twice)")
    ds = ds.isel(time=~dup)
    runs.append(ds.assign(HIST_JOINT=hist).expand_dims(experiment=[exp]))
bins = [r.attrs[f"bins_{c}"] for r in runs for c in COORDS]
assert len(set(bins[0::2])) == 1 and len(set(bins[1::2])) == 1, "experiments use different bins"

sums = xr.concat(runs, dim="experiment", join="outer").load()
EXPS = list(sums.experiment.values)
z = sums.z.values
dth = {c: float(sums[c][1] - sums[c][0]) for c in COORDS}
sums

## Time series
The largest $|\Psi|$ over $(z, \theta)$ at each time, and the largest cloud fraction, to check that the runs
are in equilibrium and to choose `T_START`, `T_END`.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for exp in EXPS:
    s = sums.sel(experiment=exp).dropna("time", how="all", subset=["RHO"])
    for c, ls in zip(COORDS, ["-", "--"]):
        psi = iso.streamfunction(s[f"MF_{c}"].values, s[f"AREA_{c}"].values)
        axs[0].plot(s.time, np.nanmax(np.abs(psi), axis=(1, 2)), ls=ls, label=f"{exp}, {LABEL[c]}")
    axs[1].plot(s.time, s.CLD_FRAC.max("z"), label=exp)
for t in (T_START, T_END):
    if t is not None:
        for ax in axs:
            ax.axvline(t, color="k", lw=0.5)
axs[0].set_ylabel(r"$\max |\Psi|$ (kg m$^{-2}$ s$^{-1}$)")
axs[1].set_ylabel("max cloud fraction")
for ax in axs:
    ax.set_xlabel("time (d)")
    ax.legend(fontsize=8)
plt.show()

## Time means
Sums over the averaging period, then the diagnostics. $\Psi$ is evaluated at the upper edge of each bin;
$\Psi_{cld}$ and $\Psi_{clr} = \Psi - \Psi_{cld}$ are both divided by the total area, so they add up to $\Psi$.

In [ ]:
tsum = sums.sel(time=slice(T_START, T_END)).sum("time", skipna=True)
nt = {exp: int(np.isfinite(sums.RHO.sel(experiment=exp, time=slice(T_START, T_END)).isel(z=0)).sum())
      for exp in EXPS}
print("snapshots in the time mean:", nt)

tm = {}
for c in COORDS:
    mf, area, mf_cld = (tsum[f"{v}_{c}"].values for v in ("MF", "AREA", "MF_CLD"))
    a_tot = area.sum(-1)
    ts = iso.two_stream(mf, area)
    with np.errstate(invalid="ignore", divide="ignore"):
        frac = np.where(area > 0, tsum[f"AREA_CLD_{c}"].values / area, np.nan)
    tm[c] = dict(
        psi=iso.streamfunction(mf, area),                       # (experiment, z, theta)
        psi_cld=iso.streamfunction(mf_cld, area),
        mean=tsum[f"ATH_{c}"].values.sum(-1) / a_tot,          # mean coordinate profile
        M_up=ts["M_up"], M_down=ts["M_down"],
        mf_cld=mf_cld.sum(-1) / a_tot, mf_all=mf.sum(-1) / a_tot,
        cld_frac_bin=frac, cld_frac=tsum[f"AREA_CLD_{c}"].values.sum(-1) / a_tot,
        th=sums[c].values + dth[c] / 2,                          # bin upper edges, where Psi is defined
    )
    tm[c]["psi_clr"] = tm[c]["psi"] - tm[c]["psi_cld"]
lim = max(np.nanmax(np.abs(tm[c]["psi"])) for c in COORDS)
levels = np.linspace(-lim, lim, 20)
xlim = (min(float(sums[c][0]) for c in COORDS), max(float(sums[c][-1]) for c in COORDS))

## Streamfunction in $\theta_e$ and frozen $\theta_e$
Top: time-mean $\Psi$ with the mean profile of each coordinate, on the same colour scale and $\theta$ axis.
Bottom: two-stream ascending and descending mass fluxes, and $\max_\theta |\Psi|$.

In [ ]:
for e, exp in enumerate(EXPS):
    fig, axs = plt.subplots(2, 2, figsize=(12, 9), sharey=True, layout="constrained")
    for ax, c in zip(axs[0], COORDS):
        pc = ax.pcolormesh(tm[c]["th"], z / 1e3, tm[c]["psi"][e], cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
        ax.contour(tm[c]["th"], z / 1e3, tm[c]["psi"][e], levels=levels, colors="k", linewidths=0.4)
        ax.plot(tm[c]["mean"][e], z / 1e3, "k", lw=2, label=f"mean {LABEL[c]}")
        ax.set_xlabel(LABEL[c] + " (K)")
        ax.set_xlim(xlim)
        ax.legend(loc="upper right")
    axs[0, 0].set_ylabel("z (km)")
    fig.colorbar(pc, ax=axs[0], label=r"$\Psi$ (kg m$^{-2}$ s$^{-1}$)")

    ax = axs[1, 0]
    for c, ls in zip(COORDS, ["-", "--"]):
        ax.plot(tm[c]["M_up"][e], z / 1e3, "C0", ls=ls, label=r"$M^+$, " + LABEL[c])
        ax.plot(-tm[c]["M_down"][e], z / 1e3, "C1", ls=ls, label=r"$-M^-$, " + LABEL[c])
    ax.set_xlabel(r"mass flux (kg m$^{-2}$ s$^{-1}$)")
    ax.set_ylabel("z (km)")
    ax.legend()

    ax = axs[1, 1]
    for c, ls in zip(COORDS, ["-", "--"]):
        ax.plot(np.nanmax(np.abs(tm[c]["psi"][e]), axis=-1), z / 1e3, "k", ls=ls, label=LABEL[c])
    ax.set_xlabel(r"$\max_\theta |\Psi|$ (kg m$^{-2}$ s$^{-1}$)")
    ax.legend()
    fig.suptitle(f"{exp}: time mean over {nt[exp]} snapshots")
    plt.show()

## $\theta_e$ versus frozen $\theta_e$
Time-mean profiles of the two coordinates, their difference, and their joint distribution over all grid
points of all processed snapshots (the joint distribution is not restricted to the averaging period).

In [ ]:
edges = {c: np.concatenate([sums[c].values - dth[c] / 2, [sums[c].values[-1] + dth[c] / 2]]) for c in COORDS}
for e, exp in enumerate(EXPS):
    fig, axs = plt.subplots(1, 3, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 1, 1.3]}, layout="constrained")
    for c, ls in zip(COORDS, ["-", "--"]):
        axs[0].plot(tm[c]["mean"][e], z / 1e3, "k", ls=ls, label=LABEL[c])
    axs[0].set_xlabel("K")
    axs[0].set_ylabel("z (km)")
    axs[0].legend()
    axs[0].set_title("time-mean profiles")

    axs[1].plot(tm["theta_ef"]["mean"][e] - tm["theta_e"]["mean"][e], z / 1e3, "k")
    axs[1].set_xlabel(r"$\theta_{ef} - \theta_e$ (K)")
    axs[1].set_title("difference of the mean profiles")

    h = sums.HIST_JOINT.sel(experiment=exp).values
    pc = axs[2].pcolormesh(edges["theta_e"], edges["theta_ef"], np.where(h > 0, h, np.nan).T, norm="log")
    axs[2].plot(xlim, xlim, "r", lw=0.8, label="1:1")
    axs[2].set_xlim(edges["theta_e"][[0, -1]])
    axs[2].set_xlabel(r"$\theta_e$ (K)")
    axs[2].set_ylabel(r"$\theta_{ef}$ (K)")
    axs[2].legend(loc="upper left")
    axs[2].set_title("joint distribution")
    fig.colorbar(pc, ax=axs[2], label="number of grid points")
    fig.suptitle(exp)
    plt.show()

## Cloudy and clear air
For each coordinate (rows): $\Psi$ and its cloudy and clear parts on the same colour scale, the fraction of
cloudy points in each $(z, \theta)$ bin, the net mass flux carried by cloudy and by clear air
($\sum_\theta$ MF / total area), and the cloud fraction profile. Cloudy means QN > `qn_cloud` (cloud water
and ice only).

In [ ]:
for e, exp in enumerate(EXPS):
    fig, axs = plt.subplots(len(COORDS), 6, figsize=(24, 5 * len(COORDS)), sharey=True, squeeze=False,
                            gridspec_kw={"width_ratios": [3, 3, 3, 3, 2, 1.2]}, layout="constrained")
    for row, c in zip(axs, COORDS):
        lim_e = np.nanmax(np.abs(tm[c]["psi"][e]))
        for ax, part in zip(row[:3], ["psi", "psi_cld", "psi_clr"]):
            pc = ax.pcolormesh(tm[c]["th"], z / 1e3, tm[c][part][e], cmap="RdBu_r", vmin=-lim_e, vmax=lim_e,
                               shading="auto")
            ax.contour(tm[c]["th"], z / 1e3, tm[c][part][e], levels=np.linspace(-lim_e, lim_e, 20),
                       colors="k", linewidths=0.4)
            ax.set_title({"psi": r"$\Psi$, all", "psi_cld": r"$\Psi$, cloudy", "psi_clr": r"$\Psi$, clear"}[part])
        fig.colorbar(pc, ax=row[2], label=r"$\Psi$ (kg m$^{-2}$ s$^{-1}$)")
        pc = row[3].pcolormesh(sums[c], z / 1e3, tm[c]["cld_frac_bin"][e], cmap="Blues", vmin=0, vmax=1, shading="auto")
        row[3].set_title("fraction of cloudy points")
        fig.colorbar(pc, ax=row[3])
        for ax in row[:4]:
            ax.set_xlabel(LABEL[c] + " (K)")
            ax.set_xlim(float(sums[c][0]), float(sums[c][-1]))
        row[0].set_ylabel("z (km)")

        row[4].plot(tm[c]["mf_cld"][e], z / 1e3, label="cloudy")
        row[4].plot(tm[c]["mf_all"][e] - tm[c]["mf_cld"][e], z / 1e3, label="clear")
        row[4].plot(tm[c]["mf_all"][e], z / 1e3, "k:", label="total")
        row[4].set_xlabel(r"net mass flux (kg m$^{-2}$ s$^{-1}$)")
        row[4].legend(loc="upper right")
        row[5].plot(tm[c]["cld_frac"][e], z / 1e3, "k")
        row[5].set_xlabel("cloud fraction")
    fig.suptitle(f"{exp}: cloudy (QN > {float(sums.attrs.get('qn_cloud', 0.01))} g/kg) and clear air, "
                 f"time mean over {nt[exp]} snapshots")
    plt.show()

## Comparison of the experiments
Overturning $\max_\theta |\Psi|$, ascending mass flux $M^+$, the part of $\max_\theta |\Psi_{cld}|$
carried by cloudy air, the mean $\theta_e$ profile, and the cloud fraction.

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(20, 5), sharey=True, layout="constrained")
for e, exp in enumerate(EXPS):
    color = f"C{e}"
    for c, ls in zip(COORDS, ["-", "--"]):
        lab = f"{exp}, {LABEL[c]}"
        axs[0].plot(np.nanmax(np.abs(tm[c]["psi"][e]), axis=-1), z / 1e3, color, ls=ls, label=lab)
        axs[1].plot(tm[c]["M_up"][e], z / 1e3, color, ls=ls, label=lab)
        axs[2].plot(np.nanmax(np.abs(tm[c]["psi_cld"][e]), axis=-1), z / 1e3, color, ls=ls, label=lab)
    axs[3].plot(tm["theta_e"]["mean"][e], z / 1e3, color, label=exp)
    axs[4].plot(tm["theta_e"]["cld_frac"][e], z / 1e3, color, label=exp)
axs[0].set_xlabel(r"$\max_\theta |\Psi|$ (kg m$^{-2}$ s$^{-1}$)")
axs[1].set_xlabel(r"$M^+$ (kg m$^{-2}$ s$^{-1}$)")
axs[2].set_xlabel(r"$\max_\theta |\Psi_{cld}|$ (kg m$^{-2}$ s$^{-1}$)")
axs[3].set_xlabel(r"mean $\theta_e$ (K)")
axs[4].set_xlabel("cloud fraction")
axs[0].set_ylabel("z (km)")
axs[0].legend(fontsize=8)
axs[3].legend(fontsize=8)
plt.show()

## Save the time-mean diagnostics

In [ ]:
out = xr.Dataset(coords={"experiment": EXPS, "z": z, **{c: sums[c].values for c in COORDS}})
for c in COORDS:
    dims = ("experiment", "z", c)
    for key in ("psi", "psi_cld", "psi_clr"):
        out[f"{key.upper()}_{c}"] = (dims, tm[c][key])
    for key in ("mean", "M_up", "M_down", "cld_frac"):
        out[f"{key.upper()}_{c}"] = (("experiment", "z"), tm[c][key])
out.attrs.update(t_start=str(T_START), t_end=str(T_END), note="PSI is evaluated at the upper edge of each bin")
out.to_netcdf(SUMS_DIR / "iso_diagnostics_time_mean.nc")
out